# ✈️ Velox: AI-Powered Airline Revenue Management Dashboard

---

## Portfolio Showcase for Data Science Manager Position
**Qatar Airways Revenue Management Team | Doha, Qatar**

---

### 👤 About This Project

This notebook documents **Velox**, a comprehensive airline revenue management dashboard I designed and built to demonstrate end-to-end data science leadership capabilities. The project directly addresses the core competencies required for the **Manager Data Science** role:

| JD Requirement | ✅ Demonstrated in Velox |
|----------------|-------------------------|
| Predictive Modeling & Demand Forecasting | S-Curve booking pace forecasting, time-series demand prediction |
| Pricing Algorithms & Optimization | Price elasticity analysis, revenue uplift simulation |
| Event-Driven Controller (SurgeSense) | Event-aware demand spikes, scenario sliders, optimization |
| AI/ML Implementation | Gemini-powered RM Assistant, RAG with faithfulness scoring |
| Cross-Functional Collaboration | Business rules engine integrating commercial, pricing, finance logic |
| Production Data Engineering | Schema design, API services, React+TypeScript+Vite stack |

---

## 🎯 The Problem Statement

### Core Challenge: Event-Driven Revenue Management

> **"Anticipating demand spikes from Doha events (World Cup legacy, Expo) or disruptions (fuel shocks) to prevent over/under-pricing."**

Modern airline revenue management faces a fundamental tension:

1. **Information Overload** — RM Analysts drown in data from PNRs, competitor scrapers, and search logs
2. **Static Forecasts** — Traditional models miss event-driven demand spikes until it's too late
3. **Decision Latency** — By the time an analyst spots an opportunity, seats have already sold at suboptimal prices

### Solution: Velox Dashboard

A **real-time decision support system** that synthesizes booking data, competitor intelligence, and operational costs into actionable pricing recommendations.

---

## 📊 Dashboard Modules & Visuals

The dashboard consists of **8 interconnected views**, each solving a specific revenue management question.

### Module Overview

| Visual | Name | Business Question |
|--------|------|------------------|
| A | Load Factor & RASK Gauges | Is this route filling seats at target revenue? |
| B | Booking Pace (S-Curve) | Are bookings ahead/behind forecast? |
| C | Competitor Price Tracker | Are we priced competitively vs market? |
| D | Route Profitability Waterfall | What's driving profit/loss on this route? |
| E | Price Elasticity Scatter | How sensitive is demand to price changes? |
| F | Overbooking Risk Histogram | How aggressive can we overbook safely? |
| G/H | RAG Faithfulness & Sources | Can we trust the AI recommendations? |

---

## 🔬 Visual A: Load Factor & RASK Gauges

### Purpose
Radial progress bars and sparklines showing current performance vs. targets for:
- **Load Factor** (% seats filled)
- **RASK** (Revenue per Available Seat Kilometer)
- **Yield** (Revenue per Passenger Kilometer)

### The Model
```python
# Efficiency Index Check
efficiency_index = RASK / (Yield * Load_Factor)

# Dilution Alert
if load_factor > 0.90 and RASK < target_RASK:
    alert = "High Demand / Low Price - Underpricing Detected"

# Trend Prediction (7-day rolling regression)
slope = linear_regression(rask_7d).slope
month_end_landing = current_rask + (slope * days_remaining)
```

### Revenue Impact Scenario

| Stage | Description |
|-------|-------------|
| **Problem** | DOH-JFK shows 95% Load Factor but Yield is 10% below budget |
| **Action** | Algorithm flags 'High Demand / Low Price'. Analyst closes lower fare buckets |
| **Uplift** | Recaptures **~$15,000 per flight** by forcing late-bookers into premium classes |

---

## 🔬 Visual B: Booking Pace (S-Curve Forecasting)

### Purpose
Multi-line chart comparing:
- **Actual** (Current cumulative bookings)
- **Forecast** (Predicted booking path)
- **Last Year** (Historical reference)

### The Model
```python
# S-Curve Logistic Function
def booking_forecast(t, L, k, t0):
    """P(t) = L / (1 + e^(-k(t-t0)))"""
    return L / (1 + np.exp(-k * (t - t0)))

# Velocity Monitoring (1st derivative)
pickup_rate = np.diff(cumulative_bookings)

# Variance Alert
if actual > forecast + 1.5 * std_dev:
    alert = "Premature Sell-Out Risk"
```

### Revenue Impact Scenario

| Stage | Description |
|-------|-------------|
| **Problem** | DOH-LHR bookings are 20% ahead of Last Year at 60 days out |
| **Action** | Model identifies sell-out risk → Restrict discount classes (O, Q, T) |
| **Uplift** | Saves last 30 seats for business travelers paying 3x fare: **+$25k revenue** |

---

## 🔬 Visual C: Competitor Price Tracker

### Purpose
Line chart overlaying Qatar Airways' lowest filed fare vs:
- Primary Competitor (Emirates/Etihad)
- Market Average

### The Model
```python
# Market Position Index (MPI)
MPI = our_price / competitor_price

# Price Following Logic
if competitor_drops_price and our_load_factor < threshold:
    action = "Match competitor fare"
elif competitor_drops_price and our_load_factor >= threshold:
    action = "Hold price - demand is strong"

# Brand Premium Estimation
brand_premium = historical_mean(our_price - market_avg)
```

### Revenue Impact Scenario

| Stage | Description |
|-------|-------------|
| **Problem** | Competitor drops fares by $50 on DOH-BKK |
| **Action** | Tracker shows our Load Factor is steady → Hold Price |
| **Uplift** | Avoids price war, saves **$10,000** (200 pax × $50) |

---

## 🔬 Visual D: Route Profitability Waterfall

### Purpose
Floating bar chart decomposing Net Profit:
- Base Fare Revenue → (+) Fuel Surcharge → (+) Ancillaries → (-) Spoilage → (-) Ops Cost → **Net Profit**

### The Model
```python
# Contribution Margin Analysis
contribution_margin = revenue - variable_costs

# Spoilage Costing (Opportunity Cost)
spoilage_cost = empty_seats * avg_marginal_revenue_of_lowest_fare

# Ancillary Penetration Rate
ancillary_rate = ancillary_revenue / total_pax
```

### Revenue Impact Scenario

| Stage | Description |
|-------|-------------|
| **Problem** | DOH-LOS has high revenue but breaks even (15 unsold Business seats) |
| **Action** | Waterfall highlights Spoilage > Upgrade cost → Trigger aggressive upgrades at T-24h |
| **Uplift** | Sells 10 upgrades at $400 each: **+$4,000 pure profit** |

---

## 🔬 Visual E: Price Elasticity Analysis

### Purpose
Scatter plot where:
- X-Axis: Price Change (%)
- Y-Axis: Revenue Uplift (%)
- Green = Profitable scenarios, Red = Loss scenarios

### The Model
```python
# Price Elasticity of Demand (PED)
# ε = (% Change in Quantity) / (% Change in Price)
# Log-Log Regression: log(Q) = α + ε*log(P)

from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(np.log(prices), np.log(quantities))
elasticity = model.coef_[0]  # e.g., -1.8 (elastic)

# Revenue Optimization
# At ε = -1, revenue is maximized (iso-revenue curve)
if elasticity < -1:  # Elastic demand
    action = "Price reduction increases total revenue"
elif elasticity > -1:  # Inelastic demand
    action = "Price increase increases total revenue"
```

### Revenue Impact Scenario

| Stage | Description |
|-------|-------------|
| **Problem** | DOH-ZAG load factor is weak (65%), analyst considers 10% promo |
| **Action** | Model shows PED = -1.8 → 10% drop predicts 18% volume increase |
| **Uplift** | Revenue = 0.9 × 1.18 = 1.062 → **+6.2% incremental revenue** |

---

## 🔬 Visual F: Overbooking Risk Optimization

### Purpose
Bar chart showing Net Financial Result for different overbooking levels:
- Green Bars: Revenue Gained > Compensation Cost
- Red Bars: Compensation Cost > Revenue Gained

### The Model
```python
# Newsvendor Model
# Balance: Cost of Underage (Empty Seat) vs Cost of Overage (Denied Boarding)

Cu = avg_fare  # Cost of not selling (spoilage)
Co = denied_boarding_compensation  # EU261 compensation
optimal_fill = Cu / (Cu + Co)  # Critical ratio

# Monte Carlo Simulation (10,000 iterations)
def simulate_overbooking(capacity, overbook, no_show_rate, n_sims=10000):
    results = []
    for _ in range(n_sims):
        show_ups = np.random.binomial(capacity + overbook, 1 - no_show_rate)
        denied = max(0, show_ups - capacity)
        revenue = min(show_ups, capacity) * avg_fare - denied * dbc_cost
        results.append(revenue)
    return np.mean(results), np.percentile(results, 5)  # EMV and VaR
```

### Revenue Impact Scenario

| Stage | Description |
|-------|-------------|
| **Problem** | Flight is full (300/300), historical no-show rate is 5% |
| **Action** | Model predicts 99% chance <296 pax show → Authorize +5 seats |
| **Uplift** | 5 extra seats × $800 = $4,000 revenue, comp cost <$200: **Net +$3,800** |

---

## 🔬 Visual G/H: AI Assistant with RAG Faithfulness

### Purpose
- **Gauge (0-100%)**: Trust score for AI recommendations
- **Sources Panel**: Documents cited with relevance scores

### The Model
```python
# RAG Pipeline with Faithfulness Scoring

# 1. Retrieval (Cosine Similarity)
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer('all-MiniLM-L6-v2')

query_embedding = encoder.encode(user_query)
doc_embeddings = encoder.encode(knowledge_base_chunks)
similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
top_k_indices = np.argsort(similarities)[-3:]  # Top 3 sources

# 2. Generation with Context
context = "\n".join([knowledge_base_chunks[i] for i in top_k_indices])
response = gemini.generate(f"Context: {context}\n\nQuestion: {user_query}")

# 3. Faithfulness Scoring (NLI-based)
# Checks: Is the generated answer entailed by the source context?
faithfulness_score = entailment_model.score(context, response)

# 4. Citation Precision
citation_precision = sentences_with_source_support / total_sentences
```

### Revenue Impact Scenario

| Stage | Description |
|-------|-------------|
| **Problem** | Agent asks if a 'Promo Fare' allows refunds - policies are complex |
| **Action** | RAG retrieves exact clause: "Rule 4.2: Non-refundable" (98% match) |
| **Uplift** | Prevents incorrect refund of non-ref ticket: **Protects $1,000+ per case** |

---

## 🗃️ Data Sources & Schema

The dashboard is powered by 5 core data tables designed to mirror real airline systems:

### 1. FACT_BOOKING_LOGS
**Transactional grain of every ticket issued (PNR data)**

| Column | Description |
|--------|-------------|
| `pnr_locator` | Unique booking reference |
| `flight_id` | Flight identifier (QR701_20260128) |
| `route_od` | Origin-Destination pair |
| `booking_date_utc` | When ticket was purchased |
| `flight_date_utc` | Departure date |
| `fare_bucket` | RBD (A-Z fare class) |
| `cabin` | Business/Economy |
| `total_fare_usd` | Ticket price |

In [ ]:
# Sample booking log entry
booking_sample = {
    "pnr_locator": "A7X8J9",
    "flight_id": "QR701_20260128",
    "route_od": "DOH-JFK",
    "booking_date_utc": "2025-11-15T08:45:00Z",
    "flight_date_utc": "2026-01-28T01:30:00Z",
    "fare_bucket": "J",
    "cabin": "Business",
    "total_fare_usd": 4250.00,
    "pos_country": "US",
    "pax_status": "ticketed"
}
print("Sample Booking Log Entry:")
booking_sample

### 2. FACT_COMPETITOR_FARES
**Daily scraped lowest available fares from competitors**

| Column | Description |
|--------|-------------|
| `scrape_date` | Date of fare collection |
| `route_od` | O&D pair |
| `airline_code` | EK, EY, etc. |
| `lowest_economy_usd` | Cheapest economy fare |
| `lowest_business_usd` | Cheapest business fare |
| `is_flash_sale` | Boolean for promotional pricing |

### 3. DIM_PAX_PROFILE
**Anonymized CRM attributes for no-show prediction**

| Column | Description |
|--------|-------------|
| `pax_hash_id` | Anonymized passenger ID |
| `loyalty_tier` | Burgundy/Silver/Gold/Platinum |
| `historical_no_show_rate` | Past no-show behavior |
| `connection_time_mins` | Inbound connection time |
| `predicted_show_prob` | ML-predicted show probability |
| `risk_category` | High/Medium/Low Risk |

### 4. FACT_SEARCH_LOGS
**Upper funnel demand (GDS/Website queries)**

| Column | Description |
|--------|-------------|
| `timestamp_utc` | Search timestamp |
| `origin` / `destination` | O&D searched |
| `conversion_event` | Did search convert to booking? |
| `error_code` | NO_AVAILABILITY = Spill demand |
| `demand_type` | Captured/Spill/Lost |

### 5. FACT_FLIGHT_COSTS
**Operating economics per flight leg**

| Column | Description |
|--------|-------------|
| `flight_id` | Flight identifier |
| `aircraft_type` | B777-300ER, A350, etc. |
| `block_hours` | Flight duration |
| `fuel_cost_usd` | Fuel expense |
| `crew_cost_usd` | Crew allocation |
| `est_spoilage_usd` | Estimated empty seat cost |

In [ ]:
# Sample flight cost entry
flight_cost = {
    "flight_id": "QR701_20260128",
    "aircraft_type": "B777-300ER",
    "block_hours": 14.5,
    "total_fuel_kg": 105000,
    "fuel_cost_usd": 78000,
    "crew_cost_usd": 12000,
    "catering_cost_usd": 8500,
    "overflight_fees_usd": 15000,
    "est_spoilage_usd": 4000
}
print("Sample Flight Cost Entry:")
flight_cost

---

## 🤖 AI/ML Models Implemented

### Model 1: Demand Forecasting (S-Curve)
- **Type**: Time-series logistic regression
- **Input**: Historical booking curves, seasonality indices
- **Output**: Predicted cumulative bookings by days-out
- **Tech**: Python (scikit-learn, statsmodels)

### Model 2: No-Show Predictor
- **Type**: Classification (Random Forest / XGBoost)
- **Input**: Passenger profile, connection data, loyalty tier
- **Output**: Show probability (0-1)
- **Tech**: Python (XGBoost), Monte Carlo simulation

### Model 3: Price Elasticity Estimator
- **Type**: Log-log regression
- **Input**: Historical price/volume pairs by segment
- **Output**: Elasticity coefficient (ε)
- **Tech**: Python (scikit-learn)

### Model 4: RM Assistant (RAG)
- **Type**: Retrieval-Augmented Generation
- **Input**: User query + Knowledge base (policies, reports)
- **Output**: Grounded answer with citations
- **Tech**: Google Gemini API, Sentence Transformers

---

## 🛠️ Technical Stack

### Frontend
- **React 19** + **TypeScript** for type-safe component development
- **Vite 6** for fast build tooling
- **TailwindCSS** for responsive, dark-mode UI
- **Recharts** for interactive data visualizations

### Data Layer
- **TanStack React Query** for server state management
- **Mock API** designed as drop-in replacement for production APIs
- **Zod** for runtime data validation

### AI Integration
- **Google Gemini API** for conversational RM assistant
- **RAG Pipeline** with faithfulness scoring

### Routes in Focus
Selected high-potential routes for demonstration:
- **DOH-SFO**: High growth market (Tech corridor)
- **DOH-JFK**: Premium leisure/business mix
- **DOH-LOS**: High yield, volatile load (Africa corridor)
- **DOH-PVG**: Emerging market, high growth potential
- **DOH-ZAG**: Event-driven demand (World Cup legacy)

---

## 🎯 Alignment with JD: SurgeSense Orchestrator

The job description highlights the **SurgeSense Orchestrator** project. Here's how Velox addresses each requirement:

| SurgeSense Requirement | Velox Implementation |
|------------------------|---------------------|
| Event-driven demand controller | Revenue Uplift Simulator with scenario sliders |
| Time-series ML (LSTM, Prophet, XGBoost) | S-Curve booking pace forecasting |
| Historical booking data | FACT_BOOKING_LOGS schema |
| Sliders for seasonality, fuel, events | Interactive simulation controls |
| Optimized pricing recommendations | Price elasticity + overbooking optimization |

In [ ]:
# Demonstration: Revenue Uplift Simulation Logic

def simulate_revenue_uplift(current_forecast, uplift_percent, price_elasticity=-1.5):
    """
    Simulate the revenue impact of a price/capacity adjustment.
    
    Args:
        current_forecast: Base revenue forecast
        uplift_percent: Price/capacity change (0-15%)
        price_elasticity: PED coefficient (default -1.5 = elastic)
    
    Returns:
        Dictionary with projected delta and recommendation
    """
    # Revenue delta based on elasticity
    volume_change = 1 + (price_elasticity * uplift_percent / 100)
    price_change = 1 + (uplift_percent / 100)
    revenue_multiplier = price_change * volume_change
    
    projected_delta = current_forecast * (revenue_multiplier - 1)
    
    recommendation = "Safe to implement" if uplift_percent <= 10 else "Monitor for spill"
    
    return {
        "uplift_percent": uplift_percent,
        "projected_delta_usd": round(projected_delta, 2),
        "recommendation": recommendation,
        "elasticity_used": price_elasticity
    }

# Example: 8% price increase on DOH-SFO
result = simulate_revenue_uplift(100000, 8, -1.5)
print("Revenue Uplift Simulation:")
result

---

## 📈 Key Metrics & KPIs Tracked

| Metric | Definition | Target |
|--------|------------|--------|
| **Load Factor** | Booked Pax / Capacity | 85-92% |
| **RASK** | Revenue / Available Seat Kilometers | Route-dependent |
| **Yield** | Revenue / Revenue Passenger Kilometers | Maximize |
| **Booking Velocity** | Daily pickup rate vs. forecast | Within ±1.5σ |
| **Market Position Index** | Our Price / Competitor Price | 0.95-1.10 |
| **Spoilage Rate** | Empty Seats / Capacity | Minimize |
| **No-Show Rate** | No-Shows / Bookings | Predict accurately |
| **RAG Faithfulness** | % answers grounded in sources | >90% |

---

## 🚀 Running the Dashboard

```bash
# Clone and install
git clone https://github.com/your-username/AirlineDashboard.git
cd AirlineDashboard
npm install --legacy-peer-deps

# Configure API key
# Create .env.local with: GEMINI_API_KEY=your_key_here

# Start development server
npm run dev
# Open http://localhost:3000
```

---

## 📞 Contact & Next Steps

This project demonstrates my capabilities in:

✅ **Predictive Modeling** — S-curve forecasting, demand prediction  
✅ **Dynamic Pricing** — Elasticity analysis, optimization  
✅ **AI/ML Implementation** — RAG, LLM integration  
✅ **Data Engineering** — Schema design, API development  
✅ **Cross-Functional Thinking** — Business rules, stakeholder alignment  
✅ **Production-Quality Code** — TypeScript, React, modern tooling

I'm excited to discuss how these skills can drive revenue optimization at **Qatar Airways**.

---

*Dashboard live at: [http://localhost:3000](http://localhost:3000)*

---